In [2]:
!uv pip install pandas scikit-learn kagglehub ipywidgets datasets transformers torch torchvision torchaudio accelerate



Using Python 3.12.13 environment at: /home/shreyas/Desktop/del_gen_ai/dlgen_ai_quiz_solver/env
Resolved 104 packages in 1.88s                                       
Uninstalled 1 package in 3ms
Installed 1 package in 6ms                                  
 - fsspec==2026.6.0
 + fsspec==2026.4.0


In [3]:
from datasets import load_dataset

In [4]:
dataset = load_dataset("csv", data_files='/home/shreyas/.cache/kagglehub/competitions/smart-mcq-solver-challenge/train.csv')

## Q1

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
        num_rows: 2000
    })
})

In [6]:
train_dataset = dataset['train']
train_dataset

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})

In [7]:
def combine_text(example):
    example['combined_text'] = example['prompt'] + ' ' + example['A']
    return example

In [8]:
train_dataset = train_dataset.map(combine_text)

In [9]:
row = train_dataset[51]
row

{'id': 52,
 'prompt': 'Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options.',
 'A': 'Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.',
 'B': 'Class L dwarfs are cooler than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are dark red in color and are brightest in infrared. Their atmosphere is cool enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to suppo

In [10]:
length = len(row["combined_text"])
length

614

## Q2

In [11]:
from transformers import AutoTokenizer,AutoModel
import torch
import transformers
import datasets

print(torch.__version__)
print(transformers.__version__)
print(datasets.__version__)

from transformers.utils import is_torch_available
print(is_torch_available())

2.12.1+cu130
5.12.1
5.0.0
True


In [12]:
import os

print(os.environ.get("USE_TORCH"))
print(os.environ.get("USE_TF"))

None
None


In [13]:
import sys
import torch
import transformers

print(sys.executable)
print(torch.__file__)
print(transformers.__file__)

/home/shreyas/Desktop/del_gen_ai/dlgen_ai_quiz_solver/env/bin/python
/home/shreyas/Desktop/del_gen_ai/dlgen_ai_quiz_solver/env/lib/python3.12/site-packages/torch/__init__.py
/home/shreyas/Desktop/del_gen_ai/dlgen_ai_quiz_solver/env/lib/python3.12/site-packages/transformers/__init__.py


In [19]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
print(tokenizer.vocab_size)

30522


## Q3

In [16]:
sep_id = tokenizer.sep_token_id
sep_id

102

## Q4

In [17]:
prompts = list(train_dataset["prompt"])

encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)

torch.Size([2000, 128])


## Q5

In [18]:
768/12

64.0

## Q6

In [20]:
prompt = train_dataset[0]["prompt"]
print(prompt)

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


In [22]:
inputs = tokenizer(prompt, return_tensors="pt")
inputs

{'input_ids': tensor([[  101,  4060,  1996,  2190,  2825,  3437,  1024,  2054,  2003,  3235,
          2002,  5178, 13327,  1005,  1055,  3193,  2006,  1996,  3276,  2090,
          2051,  1998,  2529,  4598,  1029,  2426,  1996,  3205,  7047,  1012,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]])}

In [23]:
outputs = model(**inputs)

In [24]:
print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


## Q7

In [25]:
with torch.no_grad():
    outputs = model(**inputs)

In [26]:
cls_embedding = outputs.last_hidden_state[0, 0]

In [28]:
print(round(outputs.last_hidden_state[0, 0, :5].sum().item(), 4))

-1.2001


## Q8

In [ ]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']


In [31]:
fusion_index = tokens.index("fusion")
print("Fusion token index:", fusion_index)

Fusion token index: 4


In [32]:
last_layer_head0 = outputs.attentions[-1][0, 0]

In [33]:
attention_weight = last_layer_head0[0, fusion_index].item()

print("Attention weight:", round(attention_weight, 4))

Attention weight: 0.1025


## Q9

In [35]:
!uv pip install sentence-transformers

Using Python 3.12.13 environment at: /home/shreyas/Desktop/del_gen_ai/dlgen_ai_quiz_solver/env
Resolved 59 packages in 921ms                                        
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)--------------------     0 B/582.43 KiB         
⠙ Preparing packages... (0/1)-------------------- 16.00 KiB/582.43 KiB       
⠙ Preparing packages... (0/1)-------------------- 32.00 KiB/582.43 KiB       
⠙ Preparing packages... (0/1)-------------------- 48.00 KiB/582.43 KiB       
⠙ Preparing packages... (0/1)-------------------- 60.97 KiB/582.43 KiB       
⠙ Preparing packages... (0/1)-------------------- 76.97 KiB/582.43 KiB       
⠙ Preparing packages... (0/1)-------------------- 92.97 KiB/582.43 KiB       
⠙ Preparing packages... (0/1)-------------------- 108.97 KiB/582.43 KiB      
⠙ Preparing packages... (0/1)-------------------- 124.97 KiB/582.43 KiB      
⠙ Preparing packages... (0/1)-------------------- 14

In [36]:
from sentence_transformers import SentenceTransformer, util

In [37]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
prompt = train_dataset[0]["prompt"]
option_b = train_dataset[0]["B"]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [38]:
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

In [39]:
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(round(similarity.item(), 4))

0.7658


## Q10

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [40]:
options = ["A", "B", "C", "D", "E"]

tfidf_scores = []
minilm_scores = []
improved_count = 0


In [41]:
def map_at_3(preds, answer):
    if answer == preds[0]:
        return 1.0
    elif answer == preds[1]:
        return 0.5
    elif answer == preds[2]:
        return 1/3
    return 0.0

In [ ]:
for row in train_dataset:

    prompt = row["prompt"]
    candidates = [row[o] for o in options]
    answer = row["answer"]

    corpus = [prompt] + candidates

    tfidf = TfidfVectorizer()
    X = tfidf.fit_transform(corpus)

    sims = cosine_similarity(X[0], X[1:]).flatten()

    tfidf_rank = np.argsort(sims)[::-1]
    tfidf_top3 = [options[i] for i in tfidf_rank[:3]]

    tfidf_scores.append(map_at_3(tfidf_top3, answer))

    embeddings = model.encode(corpus, convert_to_tensor=True)

    sims = util.cos_sim(
        embeddings[0],
        embeddings[1:]
    )[0].cpu().numpy()

    minilm_rank = np.argsort(sims)[::-1]
    minilm_top3 = [options[i] for i in minilm_rank[:3]]

    minilm_scores.append(map_at_3(minilm_top3, answer))

    if answer not in tfidf_top3 and answer in minilm_top3:
        improved_count += 1


In [46]:
print("MiniLM MAP@3:", round(np.mean(minilm_scores), 6))
print("Improved Count:", improved_count)

MiniLM MAP@3: 0.423083
Improved Count: 564


## Q11

In [47]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [48]:
prompt = train_dataset[1]["prompt"]

candidate_labels = [
    train_dataset[1]["A"],
    train_dataset[1]["B"],
    train_dataset[1]["C"],
]

result = classifier(prompt, candidate_labels)

print("Top Label:", result["labels"][0])
print("Score:", round(result["scores"][0], 4))

Top Label: Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.
Score: 0.4575


## Q12

In [49]:
softmax_result = classifier(
    prompt,
    candidate_labels,
    multi_label=False
)

sigmoid_result = classifier(
    prompt,
    candidate_labels,
    multi_label=True
)

softmax_sum = sum(softmax_result["scores"])
sigmoid_sum = sum(sigmoid_result["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax Sum:", softmax_sum)
print("Sigmoid Sum:", sigmoid_sum)
print("Absolute Difference:", round(difference, 4))

Softmax Sum: 0.9999999105930328
Sigmoid Sum: 0.0005096009372209664
Absolute Difference: 0.9995


## Q13

In [52]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

In [53]:
text = (
    f"Question: {train_dataset[0]['prompt']}. "
    f"Is the correct answer A: {train_dataset[0]['A']} "
    f"or B: {train_dataset[0]['B']}? "
    f"Answer with just the letter A or B."
)

In [54]:
result = generator(
    text,
    max_new_tokens=5
)

print(result)
print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B."}]
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin H